# CART Classification – Supermarket Promo Purchase Prediction

**Objective:** Build a CART (Classification and Regression Tree) model on a custom dataset that predicts whether a supermarket customer **buys a promoted product** (`Buys_Product` = Yes / No).

**Workflow followed in this notebook**
1. Import libraries
2. Create my own dataset and explore it (unique values, class counts, total records, target, features)
3. Calculate the parent Gini impurity
4. Evaluate every possible binary split and its Gini reduction
5. Choose the best root split
6. Encode the data and train the **full (unpruned) tree** – display it
7. Build the **pruned trees** (pre-pruning and post-pruning) – display them
8. Compare, evaluate (accuracy, confusion matrix, cross-validation), read the rules and predict a new customer

**Scenario:** A supermarket puts up an A4 promo poster for a product. For each of 30 customers we record the customer type, the discount level, whether the poster was noticed, and the type of visit day – and whether the customer finally bought the product.

## Step 1 – Import the required libraries

Pandas holds the table, NumPy supports calculations, `itertools` generates the binary splits, and scikit-learn trains and evaluates the CART model.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from itertools import combinations

from sklearn.tree import DecisionTreeClassifier, plot_tree, export_text
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

pd.set_option("display.max_rows", 100)

## Step 2 – Create my own dataset

The dataset is a custom supermarket-promotion dataset with **4 categorical input features** and **1 target**. Each row is one customer.

In [ ]:
rows = [
    ('New', 'Low', 'No', 'Weekday', 'Yes'),
    ('Occasional', 'Low', 'No', 'Weekday', 'No'),
    ('Regular', 'Low', 'Yes', 'Weekend', 'Yes'),
    ('Regular', 'Low', 'No', 'Weekday', 'No'),
    ('Regular', 'Low', 'Yes', 'Weekend', 'Yes'),
    ('Regular', 'Low', 'Yes', 'Weekend', 'Yes'),
    ('Occasional', 'Low', 'No', 'Weekday', 'No'),
    ('Occasional', 'High', 'Yes', 'Weekend', 'Yes'),
    ('Occasional', 'Low', 'Yes', 'Weekday', 'No'),
    ('Occasional', 'Medium', 'No', 'Weekend', 'Yes'),
    ('New', 'Medium', 'No', 'Weekday', 'No'),
    ('Occasional', 'Low', 'Yes', 'Weekend', 'No'),
    ('New', 'High', 'No', 'Weekend', 'Yes'),
    ('Regular', 'Low', 'No', 'Weekday', 'No'),
    ('Regular', 'Medium', 'No', 'Weekday', 'No'),
    ('Regular', 'High', 'No', 'Weekend', 'Yes'),
    ('Occasional', 'Medium', 'No', 'Weekday', 'No'),
    ('New', 'Medium', 'Yes', 'Weekday', 'No'),
    ('New', 'High', 'No', 'Weekend', 'Yes'),
    ('Occasional', 'Medium', 'Yes', 'Weekend', 'Yes'),
    ('Occasional', 'Low', 'No', 'Weekday', 'No'),
    ('New', 'Low', 'Yes', 'Weekend', 'Yes'),
    ('New', 'Low', 'Yes', 'Weekend', 'Yes'),
    ('New', 'Low', 'No', 'Weekend', 'No'),
    ('New', 'High', 'No', 'Weekday', 'Yes'),
    ('Regular', 'Low', 'Yes', 'Weekday', 'Yes'),
    ('Occasional', 'Low', 'No', 'Weekend', 'Yes'),
    ('New', 'High', 'No', 'Weekend', 'Yes'),
    ('Occasional', 'High', 'Yes', 'Weekend', 'Yes'),
    ('Occasional', 'High', 'No', 'Weekend', 'Yes'),
]
columns = ["Customer_Type", "Discount_Level", "Poster_Noticed", "Visit_Day", "Buys_Product"]

df = pd.DataFrame(rows, columns=columns)
df.index = df.index + 1          # customers numbered from 1
df.index.name = "Customer_ID"
display(df)

## Step 3 – Explore the dataset

Structure, unique values in each column, class counts, total records, target variable and input features.

In [ ]:
print("Shape (rows, columns):", df.shape)
print("Total records        :", len(df))
print()
df.info()

In [ ]:
print("UNIQUE VALUES IN EACH COLUMN")
print("-" * 45)
for col in df.columns:
    print(f"{col:15s}: {list(df[col].unique())}   ({df[col].nunique()} unique)")

In [ ]:
target = "Buys_Product"
features = [c for c in df.columns if c != target]

counts = df[target].value_counts()
yes_count, no_count, total = counts["Yes"], counts["No"], len(df)

print("Target variable :", target)
print("Input features  :", features)
print(f"\nYes records: {yes_count}  |  No records: {no_count}  |  Total: {total}")

counts.plot(kind="bar", color=["#2e8b57", "#c0392b"], figsize=(4, 3), rot=0)
plt.title("Class distribution of Buys_Product"); plt.ylabel("Customers"); plt.show()

**Interpretation:** There are 30 records. The target `Buys_Product` has two classes (Yes / No), so this is a **classification** problem. The four columns `Customer_Type`, `Discount_Level`, `Poster_Noticed` and `Visit_Day` are the input features. The classes are fairly balanced (see counts above).

## Step 4 – Parent Gini impurity

Gini impurity of a node:

$$Gini = 1 - \sum_k p_k^2 = 1 - [P(Yes)^2 + P(No)^2]$$

A value of 0 means a pure node; for two classes the maximum is 0.5.

In [ ]:
def gini(labels):
    """Gini impurity of a collection of class labels."""
    p = pd.Series(labels).value_counts(normalize=True)
    return 1 - (p ** 2).sum()

parent_gini = gini(df[target])
print(f"P(Yes) = {yes_count}/{total} = {yes_count/total:.3f}")
print(f"P(No)  = {no_count}/{total} = {no_count/total:.3f}")
print(f"Parent Gini = 1 - ({yes_count/total:.3f}^2 + {no_count/total:.3f}^2) = {parent_gini:.4f}")

## Step 5 – All possible binary splits and their Gini reduction

CART always makes **binary** splits. For a feature with *k* categories there are $2^{k-1}-1$ unique ways to divide the categories into a left group and a right group.

For every split we compute:

$$Gini_{weighted} = \frac{N_L}{N}Gini_{left} + \frac{N_R}{N}Gini_{right} \qquad Gini\ Reduction = Gini_{parent} - Gini_{weighted}$$

The split with the **highest Gini reduction** (lowest weighted Gini) is the best.

In [ ]:
def binary_splits(categories):
    """All unique 2-group partitions of a list of categories."""
    categories = list(categories)
    anchor, rest = categories[0], categories[1:]       # anchor stays on the left -> no mirror duplicates
    result = []
    for r in range(len(rest) + 1):
        for extra in combinations(rest, r):
            left = [anchor, *extra]
            right = [c for c in categories if c not in left]
            if right:
                result.append((left, right))
    return result

records = []
for feat in features:
    for left_vals, right_vals in binary_splits(df[feat].unique()):
        left  = df[df[feat].isin(left_vals)][target]
        right = df[df[feat].isin(right_vals)][target]
        g_left, g_right = gini(left), gini(right)
        g_weighted = len(left)/total * g_left + len(right)/total * g_right
        records.append({
            "Feature": feat,
            "Left group": ", ".join(left_vals),
            "Right group": ", ".join(right_vals),
            "N left": len(left), "N right": len(right),
            "Gini left": round(g_left, 3), "Gini right": round(g_right, 3),
            "Weighted Gini": round(g_weighted, 3),
            "Gini reduction": round(parent_gini - g_weighted, 3),
        })

split_table = pd.DataFrame(records)
print(f"Parent Gini = {parent_gini:.3f}   |   Total candidate splits = {len(split_table)}")
display(split_table)

In [ ]:
ranked = split_table.sort_values("Gini reduction", ascending=False).reset_index(drop=True)
print("Splits ranked by Gini reduction (best first):")
display(ranked.head(5))

best = ranked.iloc[0]
print(f"BEST ROOT SPLIT -> {best['Feature']}:  [{best['Left group']}]  |  [{best['Right group']}]")
print(f"Gini reduction = {best['Gini reduction']}")

best_per_feature = split_table.loc[split_table.groupby("Feature")["Gini reduction"].idxmax()]
best_per_feature.set_index("Feature")["Gini reduction"].sort_values().plot(
    kind="barh", figsize=(6, 3), color="#3b7dd8")
plt.title("Best Gini reduction per feature"); plt.xlabel("Gini reduction"); plt.show()

**Interpretation:** The table compares every candidate split. The split at the top of the ranking removes the most impurity, so CART selects it as the **root node**. Features whose best split has a small Gini reduction are weak predictors and will be used lower in the tree (or pruned away).

## Step 6 – Encode the features

Scikit-learn trees need numbers. One-hot encoding turns each category into its own 0/1 column; the tree then splits on questions like *“Discount_Level_High ≤ 0.5?”* which is exactly a binary CART split.

In [ ]:
X = pd.get_dummies(df[features], dtype=int)
y = df[target].map({"No": 0, "Yes": 1})
class_names = ["No", "Yes"]

print("Encoded columns:", list(X.columns))
display(X.head())

## Step 7 – Train / test split

A stratified 70/30 split keeps the Yes/No ratio similar in both sets.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, stratify=y, random_state=3)

print("Training records:", len(X_train), "| Testing records:", len(X_test))
print("Train class counts:", dict(y_train.value_counts()), "| Test class counts:", dict(y_test.value_counts()))

## Step 8 – Full (unpruned) CART tree

With no restrictions the tree keeps splitting until every leaf is pure – it fits the training data perfectly but is likely to **overfit**.

In [ ]:
full_tree = DecisionTreeClassifier(criterion="gini", random_state=3)
full_tree.fit(X_train, y_train)

print("Full tree  -> depth:", full_tree.get_depth(), "| leaves:", full_tree.get_n_leaves(),
      "| nodes:", full_tree.tree_.node_count)

plt.figure(figsize=(22, 11))
plot_tree(full_tree, feature_names=X.columns, class_names=class_names,
          filled=True, rounded=True, fontsize=9)
plt.title("FULL (unpruned) CART tree – Supermarket Promo Purchase", fontsize=15)
plt.show()

**Reading the tree:** each box shows the split question, the node's Gini, the number of samples, the class counts (`value = [No, Yes]`) and the predicted class. The *left* branch means the condition is **True** (feature = 0, i.e. category absent) and the *right* branch means **False**. The root is the best split found in Step 5.

In [ ]:
print(export_text(full_tree, feature_names=list(X.columns)))

## Step 9 – Root node and child nodes

Verifying the root split by hand against scikit-learn's tree structure.

In [ ]:
t = full_tree.tree_
root_feat = X.columns[t.feature[0]]
left_id, right_id = t.children_left[0], t.children_right[0]

print("Root split question :", f"{root_feat} <= {t.threshold[0]:.1f}")
print(f"Root node   -> samples: {t.n_node_samples[0]}, Gini: {t.impurity[0]:.3f}")
print(f"Left child  -> samples: {t.n_node_samples[left_id]}, Gini: {t.impurity[left_id]:.3f}")
print(f"Right child -> samples: {t.n_node_samples[right_id]}, Gini: {t.impurity[right_id]:.3f}")

w = (t.n_node_samples[left_id]*t.impurity[left_id] + t.n_node_samples[right_id]*t.impurity[right_id]) / t.n_node_samples[0]
print(f"Weighted child Gini = {w:.3f}  |  Gini reduction = {t.impurity[0]-w:.3f}")

## Step 10 – Pruning

Pruning simplifies the tree to reduce overfitting. Two methods are used:

* **Pre-pruning** – stop growth early using `max_depth` and `min_samples_leaf`.
* **Post-pruning** – grow the full tree, then cut weak branches using **cost-complexity pruning** (`ccp_alpha`). A larger alpha prunes more.

### 10a – Pre-pruned tree

In [ ]:
pre_tree = DecisionTreeClassifier(criterion="gini", max_depth=3, min_samples_leaf=2, random_state=3)
pre_tree.fit(X_train, y_train)

print("Pre-pruned tree -> depth:", pre_tree.get_depth(), "| leaves:", pre_tree.get_n_leaves())

plt.figure(figsize=(16, 8))
plot_tree(pre_tree, feature_names=X.columns, class_names=class_names,
          filled=True, rounded=True, fontsize=10)
plt.title("PRE-PRUNED CART tree (max_depth=3, min_samples_leaf=2)", fontsize=14)
plt.show()

### 10b – Post-pruned tree (cost-complexity pruning)

First we look at the pruning path, then choose the largest `ccp_alpha` (simplest tree) whose **cross-validated** accuracy is within 0.03 of the best on the training data (the test set stays untouched).

In [ ]:
path = full_tree.cost_complexity_pruning_path(X_train, y_train)
alphas = np.unique(path.ccp_alphas)

cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=3)
alpha_table = []
for a in alphas:
    m = DecisionTreeClassifier(criterion="gini", ccp_alpha=a, random_state=3)
    alpha_table.append({
        "ccp_alpha": round(a, 4),
        "CV accuracy": cross_val_score(m, X_train, y_train, cv=cv).mean(),
        "Leaves": m.fit(X_train, y_train).get_n_leaves(),
    })
alpha_table = pd.DataFrame(alpha_table)
display(alpha_table.round(3))

# choose the SIMPLEST tree (largest alpha) whose CV accuracy is within 0.03 of the best score
top = alpha_table["CV accuracy"].max()
best_alpha = alpha_table[alpha_table["CV accuracy"] >= top - 0.03]["ccp_alpha"].max()
print("Selected ccp_alpha =", best_alpha)

In [ ]:
post_tree = DecisionTreeClassifier(criterion="gini", ccp_alpha=best_alpha, random_state=3)
post_tree.fit(X_train, y_train)

print("Post-pruned tree -> depth:", post_tree.get_depth(), "| leaves:", post_tree.get_n_leaves())

plt.figure(figsize=(14, 7))
plot_tree(post_tree, feature_names=X.columns, class_names=class_names,
          filled=True, rounded=True, fontsize=10)
plt.title(f"POST-PRUNED CART tree (ccp_alpha={best_alpha:.4f})", fontsize=14)
plt.show()

In [ ]:
print("Rules of the post-pruned tree:\n")
print(export_text(post_tree, feature_names=list(X.columns)))

## Step 11 – Compare full, pre-pruned and post-pruned trees

In [ ]:
models = {"Full tree": full_tree, "Pre-pruned tree": pre_tree, "Post-pruned tree": post_tree}
cv5 = StratifiedKFold(n_splits=5, shuffle=True, random_state=3)

comparison = pd.DataFrame([{
    "Model": name,
    "Depth": m.get_depth(),
    "Leaves": m.get_n_leaves(),
    "Train accuracy": accuracy_score(y_train, m.predict(X_train)),
    "Test accuracy": accuracy_score(y_test, m.predict(X_test)),
    "5-fold CV accuracy (all data)": cross_val_score(
        DecisionTreeClassifier(**m.get_params()), X, y, cv=cv5).mean(),
} for name, m in models.items()]).round(3)
display(comparison)

comparison.set_index("Model")[["Train accuracy", "Test accuracy"]].plot(
    kind="bar", figsize=(7, 4), rot=0, color=["#3b7dd8", "#f5a623"])
plt.ylim(0, 1.05); plt.title("Train vs Test accuracy"); plt.ylabel("Accuracy"); plt.show()

**Interpretation:** The full tree typically reaches 100 % training accuracy but is more complex, which signals overfitting. The pruned trees use fewer leaves, are easier to explain, and generally keep similar or better accuracy on unseen data (test / cross-validation). A gap between train and test accuracy shows how much a model overfits.

## Step 12 – Detailed evaluation (confusion matrix and classification report)

In [ ]:
for name, m in models.items():
    pred = m.predict(X_test)
    print("=" * 55)
    print(name.upper())
    print("=" * 55)
    print("Confusion matrix [rows = actual No/Yes, columns = predicted No/Yes]:")
    print(confusion_matrix(y_test, pred))
    print(classification_report(y_test, pred, target_names=class_names, zero_division=0))

## Step 13 – Feature importance

In [ ]:
importance = pd.Series(post_tree.feature_importances_, index=X.columns).sort_values()
importance = importance[importance > 0]
display(importance.sort_values(ascending=False).round(3).to_frame("Importance"))

importance.plot(kind="barh", figsize=(7, 4), color="#2e8b57")
plt.title("Feature importance – post-pruned tree"); plt.xlabel("Importance"); plt.show()

## Step 14 – Predict a new customer

The new record is encoded with the same columns as the training data before prediction.

In [ ]:
new_customer = pd.DataFrame([{
    "Customer_Type": "Regular", "Discount_Level": "High",
    "Poster_Noticed": "Yes", "Visit_Day": "Weekend",
}])
display(new_customer)

new_X = pd.get_dummies(new_customer, dtype=int).reindex(columns=X.columns, fill_value=0)

for name, m in models.items():
    pred = m.predict(new_X)[0]
    prob = m.predict_proba(new_X)[0]
    print(f"{name:17s} -> Buys_Product = {class_names[pred]}   (P(No)={prob[0]:.2f}, P(Yes)={prob[1]:.2f})")

## Conclusion

* A custom supermarket-promotion dataset (30 customers, 4 categorical features, target `Buys_Product`) was created and explored.
* The parent Gini impurity was computed and **every binary split** was evaluated; the split with the largest Gini reduction became the root of the CART tree.
* The **full tree** fits the training data perfectly but is large and prone to overfitting.
* The **pre-pruned** tree (depth and leaf-size limits) and the **post-pruned** tree (cost-complexity pruning, alpha chosen by cross-validation) are smaller and easier to interpret while keeping good predictive performance.
* The final model produces simple if/then rules that a store manager can read – for example, which combinations of discount level and customer type lead to a purchase.

**Limitation:** with only 30 records the accuracy values vary a lot between splits, so the results are a learning demonstration rather than a production benchmark.